In [0]:
# Output Guardrails (hand-built) + eval integration
# Paste into a NEW notebook in the workspace, run top to bottom.
#
# WHY hand-built: the managed Unity AI Gateway LLM guardrails are gated on the
# SAP Databricks trial tier (the LLMs tab is locked). So we build the same idea
# as a function, the way a team does when the managed feature is unavailable.
# The managed Gateway is noted as the production upgrade path.
#
# The guardrail has TWO layers, which is how real production does it:
#   Layer 1 (generic):  block sensitive-data (PII) leaking into an answer.
#   Layer 2 (business): enforce THIS system's honesty contract, that a journal
#                       anomaly is never stated as confirmed fraud.

# COMMAND ----------
import re

# ---------------------------------------------------------------------------
# LAYER 1 - generic safety / PII check
# Scans an output for patterns that look like sensitive data. In production the
# managed Gateway PII filter does this; here we pattern-match a few examples.
# Returns a list of issues found (empty = clean).
# ---------------------------------------------------------------------------
PII_PATTERNS = {
    "email":        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "iban_like":    r"\b[A-Z]{2}\d{2}[A-Z0-9]{10,30}\b",
    "long_number":  r"\b\d{12,19}\b",   # card / account-like long digit runs
    "ssn_like":     r"\b\d{3}-\d{2}-\d{4}\b",
}

def check_pii(text: str):
    issues = []
    for name, pattern in PII_PATTERNS.items():
        if re.search(pattern, text):
            issues.append(f"possible {name} in output")
    return issues

# ---------------------------------------------------------------------------
# LAYER 2 - business honesty contract
# The journal-risk agent must never present an anomaly as confirmed fraud, and
# when it discusses anomalies it must keep the 'statistical / not confirmed'
# framing. This guardrail enforces that contract on the output.
#
# Two checks:
#   (a) BLOCK phrases that assert confirmed wrongdoing.
#   (b) If the answer is about journal anomalies, REQUIRE the honesty framing
#       to be present.
# ---------------------------------------------------------------------------
FORBIDDEN_PHRASES = [
    "confirmed fraud", "is fraud", "fraudulent entries", "proven fraud",
    "definitely fraud", "committed fraud", "guilty of",
]

HONESTY_MARKERS = [
    "statistical", "not confirmed", "for review", "indicative",
    "worth reviewing", "not confirmed fraud", "review by",
]

ANOMALY_TOPIC_MARKERS = ["anomaly", "anomalies", "journal", "flagged"]

def check_honesty_contract(text: str):
    issues = []
    low = text.lower()

    # (a) hard block on assertions of confirmed wrongdoing,
    #     but NOT when the phrase is negated (e.g. "not confirmed fraud" is GOOD)
    for phrase in FORBIDDEN_PHRASES:
        for m in re.finditer(re.escape(phrase), low):
            start = m.start()
            preceding = low[max(0, start-12):start]   # look just before the phrase
            if "not " in preceding or "never " in preceding or "n't " in preceding:
                continue   # negated -> this is the honest framing, allow it
            issues.append(f"asserts wrongdoing: '{phrase}'")

    # (b) if the answer is about anomalies, require honesty framing
    is_about_anomalies = any(m in low for m in ANOMALY_TOPIC_MARKERS)
    has_framing = any(m in low for m in HONESTY_MARKERS)
    if is_about_anomalies and not has_framing:
        issues.append("anomaly answer missing 'statistical / for review' framing")

    return issues

# ---------------------------------------------------------------------------
# The guardrail gate: runs both layers, decides pass / block.
# In real time this wraps the agent call: agent -> guardrail -> user.
# action 'block' means do not show the raw answer; show a safe message instead.
# ---------------------------------------------------------------------------
def guardrail(output_text: str):
    pii = check_pii(output_text)
    honesty = check_honesty_contract(output_text)
    all_issues = pii + honesty
    if all_issues:
        return {
            "action": "block",
            "issues": all_issues,
            "safe_message": ("This response was held for review by the output "
                             "guardrail. Please rephrase or escalate to a human "
                             "reviewer."),
        }
    return {"action": "pass", "issues": [], "safe_message": None}

# COMMAND ----------
# Demonstrate the guardrail catching bad outputs and passing good ones.

good = ("Company code 1710 shows a 2.15% anomaly rate. These are statistical "
        "flags for review, not confirmed fraud.")

bad_fraud = ("Company code 1710 has fraudulent entries and is guilty of "
             "manipulating its journals.")

bad_missing_framing = ("Company code 1710 has 828 anomalies in its journal "
                       "entries.")   # about anomalies but no honesty framing

bad_pii = ("Contact the controller at john.doe@example.com regarding account "
           "4929123456789012.")

for label, text in [("GOOD", good),
                    ("BAD - asserts fraud", bad_fraud),
                    ("BAD - missing framing", bad_missing_framing),
                    ("BAD - PII leak", bad_pii)]:
    result = guardrail(text)
    print(f"[{label}] -> {result['action']}")
    if result["issues"]:
        for i in result["issues"]:
            print("    -", i)
    print()

# COMMAND ----------
# Integrate into the agent flow (real-time pattern).
# This is how you would wrap a call to the supervisor endpoint so EVERY answer
# passes through the guardrail before reaching the user.
#
# Pseudocode shape (adapt endpoint call to your serving client):
#
#   raw = call_supervisor_endpoint(user_question)   # the agent's answer
#   check = guardrail(raw)
#   final = raw if check["action"] == "pass" else check["safe_message"]
#   return final
#
# The point: the guardrail is not a manual step. It sits in the response path
# so it runs automatically on every answer, in milliseconds. That is what
# "real-time guardrail" means.

# COMMAND ----------
# Eval integration: add guardrail cases to the harness so the contract is
# enforced continuously, not just once. Each case feeds a known output through
# the guardrail and asserts the expected action.

guardrail_eval_cases = [
    {"name": "passes honest anomaly answer",
     "text": good, "expect": "pass"},
    {"name": "blocks confirmed-fraud assertion",
     "text": bad_fraud, "expect": "block"},
    {"name": "blocks anomaly answer with no framing",
     "text": bad_missing_framing, "expect": "block"},
    {"name": "blocks PII leak",
     "text": bad_pii, "expect": "block"},
]

passed = 0
for case in guardrail_eval_cases:
    got = guardrail(case["text"])["action"]
    ok = (got == case["expect"])
    passed += ok
    print(("PASS" if ok else "FAIL"), "-", case["name"], f"(got {got})")

print(f"\nGuardrail eval: {passed}/{len(guardrail_eval_cases)} passed")

# COMMAND ----------
# OPTIONAL: persist guardrail eval results to a Delta table, the same drift
# record pattern as the main eval harness, so guardrail performance is tracked
# over time alongside agent quality.
#
# from pyspark.sql import functions as F
# import datetime
# rows = [(c["name"], c["expect"], guardrail(c["text"])["action"],
#          datetime.datetime.now()) for c in guardrail_eval_cases]
# df = spark.createDataFrame(rows, ["case","expected","got","run_at"])
# df.write.mode("append").saveAsTable("workspace.default.guardrail_eval_results")